In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="",
    base_url="https://finderllm2-resource.services.ai.azure.com/openai/v1"
)

response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "user", "content": "Hello!"}
    ],
)

print(response.choices[0].message.content)

Hi there! How can I help you today?


In [ ]:
import os
import base64
import json
from openai import OpenAI

# ------------------------------
# CONFIG
# ------------------------------
API_KEY = ""
BASE_URL = "https://finderllm2-resource.services.ai.azure.com/openai/v1"
MODEL = "gpt-5-mini"
ROOT_FOLDER = "./Charts"

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

# ------------------------------
# IMAGE ENCODING
# ------------------------------
def encode_image(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()

# ------------------------------
# PROMPT — dual Y-axis aware
# ------------------------------
PROMPT = """
You are a vision system analyzing a financial chart.

Describe ONLY what is visually present.

For each subplot:
- number of subplots in the full image
- number of time series
- color and label of each series
- which y-axis each series belongs to: "left" or "right"
- x-axis label and visible tick values
- left y-axis label and approximate min/max
- right y-axis label and approximate min/max (only if a second y-axis is visually present)

A dual y-axis exists when:
  - there are two different y-axis scales, one on the left and one on the right side of the plot
  - the right axis has its own label or tick values distinct from the left

Return ONLY valid JSON in this exact format:

{
  "num_plots": int,
  "plots": [
    {
      "plot_id": int,
      "num_series": int,
      "dual_y_axis": bool,
      "series": [
        {
          "color": "...",
          "label": "...",
          "y_axis_side": "left" or "right"
        }
      ],
      "x_axis": {
        "label": "...",
        "values": []
      },
      "y_axis_left": {
        "label": "...",
        "min": float,
        "max": float
      },
      "y_axis_right": {
        "label": "...",
        "min": float,
        "max": float
      }
    }
  ]
}

If there is no right y-axis for a subplot, set:
  "dual_y_axis": false,
  "y_axis_right": {"label": null, "min": null, "max": null}
"""

# ------------------------------
# FOLDER HELPERS
# ------------------------------
def get_chart_folders(root):
    return [
        os.path.join(root, d)
        for d in os.listdir(root)
        if os.path.isdir(os.path.join(root, d))
    ]

def get_images(folder):
    valid_ext = (".png", ".jpg", ".jpeg", ".webp")
    return sorted(
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith(valid_ext)
    )

# ------------------------------
# SCHEMA VALIDATOR
# Catches structurally bad responses early
# ------------------------------
def validate_output(parsed):
    """
    Returns (is_valid: bool, issues: list[str])
    Checks required keys and dual_y_axis consistency.
    """
    issues = []
    if "num_plots" not in parsed:
        issues.append("missing num_plots")
    if "plots" not in parsed or not isinstance(parsed["plots"], list):
        issues.append("missing or invalid plots array")
        return False, issues

    for i, plot in enumerate(parsed["plots"]):
        pid = f"plot[{i}]"
        for key in ("plot_id", "num_series", "dual_y_axis", "series", "x_axis", "y_axis_left", "y_axis_right"):
            if key not in plot:
                issues.append(f"{pid} missing '{key}'")

        # Check series have y_axis_side
        for j, s in enumerate(plot.get("series", [])):
            if "y_axis_side" not in s:
                issues.append(f"{pid} series[{j}] missing 'y_axis_side'")

        # If dual_y_axis=True but right axis is null → inconsistency
        if plot.get("dual_y_axis") and plot.get("y_axis_right", {}).get("min") is None:
            issues.append(f"{pid} dual_y_axis=true but y_axis_right has null values")

        # If dual_y_axis=False but some series claim "right" → flag it
        if not plot.get("dual_y_axis"):
            right_series = [s for s in plot.get("series", []) if s.get("y_axis_side") == "right"]
            if right_series:
                issues.append(f"{pid} dual_y_axis=false but {len(right_series)} series assigned to right axis")

    return len(issues) == 0, issues

# ------------------------------
# PROCESS ALL CHARTS
# ------------------------------
chart_folders = get_chart_folders(ROOT_FOLDER)
print(f"Found {len(chart_folders)} chart groups\n")

results = {}

for chart_folder in chart_folders:
    chart_name = os.path.basename(chart_folder)
    print(f"\n📁 Processing: {chart_name}")

    images = get_images(chart_folder)
    print(f"   Found {len(images)} images")

    results[chart_name] = []

    for image_path in images:
        fname = os.path.basename(image_path)
        print(f"   📊 {fname}")

        image_base64 = encode_image(image_path)

        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": PROMPT},
                        {
                            "type": "image_url",
                            "image_url": {"url": f"data:image/png;base64,{image_base64}"}
                        }
                    ]
                }
            ]
        )

        raw_output = response.choices[0].message.content

        # Strip markdown code fences if model wraps output in ```json ... ```
        cleaned = raw_output.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("```")[1]
            if cleaned.startswith("json"):
                cleaned = cleaned[4:]
            cleaned = cleaned.strip()

        entry = {"image": fname, "output": None, "parse_error": None, "validation_issues": []}

        try:
            parsed = json.loads(cleaned)
            is_valid, issues = validate_output(parsed)

            entry["output"] = parsed
            entry["validation_issues"] = issues

            if issues:
                print(f"   ⚠️  Schema issues: {issues}")

            # Convenience: count how many subplots have dual Y-axis
            dual_count = sum(1 for p in parsed.get("plots", []) if p.get("dual_y_axis"))
            if dual_count:
                print(f"   🔀 {dual_count} subplot(s) with dual Y-axis detected")

        except json.JSONDecodeError as e:
            print(f"   ❌ JSON parse error: {e}")
            entry["parse_error"] = str(e)
            entry["output"] = raw_output  # keep raw so nothing is lost

        results[chart_name].append(entry)

    print("   ✅ Done")

# ------------------------------
# SAVE RESULTS
# ------------------------------
output_path = "llm_outputs_grouped_2.json"
with open(output_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"\n✅ Results saved to {output_path}")

Found 6 chart groups


📁 Processing: Chart1
   Found 10 images
   📊 ^GSPC_2025-03-10_chart1.png
   📊 ^GSPC_2025-03-11_chart1.png
   📊 ^GSPC_2025-03-12_chart1.png
   📊 ^GSPC_2025-03-13_chart1.png
   📊 ^GSPC_2025-03-14_chart1.png
   📊 ^GSPC_2025-03-17_chart1.png
   📊 ^GSPC_2025-03-18_chart1.png
   📊 ^GSPC_2025-03-19_chart1.png
   📊 ^GSPC_2025-03-20_chart1.png
   📊 ^GSPC_2025-03-21_chart1.png
   ✅ Done

📁 Processing: Chart6
   Found 10 images
   📊 ^GSPC_2025-03-10_chart6.png
   📊 ^GSPC_2025-03-11_chart6.png
   📊 ^GSPC_2025-03-12_chart6.png
   📊 ^GSPC_2025-03-13_chart6.png
   📊 ^GSPC_2025-03-14_chart6.png
   📊 ^GSPC_2025-03-17_chart6.png
   📊 ^GSPC_2025-03-18_chart6.png
   📊 ^GSPC_2025-03-19_chart6.png
   📊 ^GSPC_2025-03-20_chart6.png
   📊 ^GSPC_2025-03-21_chart6.png
   ✅ Done

📁 Processing: Chart5
   Found 10 images
   📊 ^GSPC_2025-03-10_chart5.png
   🔀 1 subplot(s) with dual Y-axis detected
   📊 ^GSPC_2025-03-11_chart5.png
   🔀 1 subplot(s) with dual Y-axis detected
   📊 ^GSPC_2025-03-12

In [ ]:
import os
import base64
import json
import re
from collections import Counter
from openai import OpenAI

# ------------------------------
# CONFIG
# ------------------------------
API_KEY = ""
BASE_URL = "https://finderllm2-resource.services.ai.azure.com/openai/v1"
MODEL = "gpt-5-mini"
ROOT_FOLDER = "./Charts"

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

def encode_image(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()

PROMPT = """
You are a vision system analyzing a financial chart.

Describe ONLY what is visually present.

For each subplot:
- number of subplots in the full image
- number of time series
- color and label of each series
- which y-axis each series belongs to: "left" or "right"
- x-axis label and visible tick values
- left y-axis label and approximate min/max
- right y-axis label and approximate min/max (only if a second y-axis is visually present)

A dual y-axis exists when:
  - there are two different y-axis scales, one on the left and one on the right side of the plot
  - the right axis has its own label or tick values distinct from the left

Return ONLY valid JSON in this exact format:

{
  "num_plots": int,
  "plots": [
    {
      "plot_id": int,
      "num_series": int,
      "dual_y_axis": bool,
      "series": [
        {
          "color": "...",
          "label": "...",
          "y_axis_side": "left" or "right"
        }
      ],
      "x_axis": {
        "label": "...",
        "values": []
      },
      "y_axis_left": {
        "label": "...",
        "min": float,
        "max": float
      },
      "y_axis_right": {
        "label": "...",
        "min": float,
        "max": float
      }
    }
  ]
}

If there is no right y-axis for a subplot, set:
  "dual_y_axis": false,
  "y_axis_right": {"label": null, "min": null, "max": null}
"""

def get_chart_folders(root):
    return [
        os.path.join(root, d)
        for d in os.listdir(root)
        if os.path.isdir(os.path.join(root, d))
    ]

def get_images(folder):
    valid_ext = (".png", ".jpg", ".jpeg", ".webp")
    return sorted(
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith(valid_ext)
    )

def validate_output(parsed):
    issues = []
    if "num_plots" not in parsed:
        issues.append("missing num_plots")
    if "plots" not in parsed or not isinstance(parsed["plots"], list):
        issues.append("missing or invalid plots array")
        return False, issues
    for i, plot in enumerate(parsed["plots"]):
        pid = f"plot[{i}]"
        for key in ("plot_id", "num_series", "dual_y_axis", "series", "x_axis", "y_axis_left", "y_axis_right"):
            if key not in plot:
                issues.append(f"{pid} missing '{key}'")
        for j, s in enumerate(plot.get("series", [])):
            if "y_axis_side" not in s:
                issues.append(f"{pid} series[{j}] missing 'y_axis_side'")
        if plot.get("dual_y_axis") and plot.get("y_axis_right", {}).get("min") is None:
            issues.append(f"{pid} dual_y_axis=true but y_axis_right has null values")
        if not plot.get("dual_y_axis"):
            right_series = [s for s in plot.get("series", []) if s.get("y_axis_side") == "right"]
            if right_series:
                issues.append(f"{pid} dual_y_axis=false but {len(right_series)} series assigned to right axis")
    return len(issues) == 0, issues

def strip_fences(raw):
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("```")[1]
        if cleaned.startswith("json"):
            cleaned = cleaned[4:]
    return cleaned.strip()

# ------------------------------
# ROLLING WINDOW CONSISTENCY
# Runs after all images in a folder are processed.
# Checks whether structural readings stay stable
# across the sequence of windows (days).
# ------------------------------
def rolling_consistency(folder_results):
    """
    Given the list of per-image results for one chart type,
    compute consistency metrics across the rolling sequence.

    Returns a dict with:
      - num_plots_stable:     True if num_plots never changes across days
      - num_series_stable:    True if num_series per subplot never changes
      - label_consistency:    fraction of windows where labels match the mode (most common set)
      - color_consistency:    same for colors
      - dual_y_stable:        True if dual_y_axis flag never flips across days
      - parse_success_rate:   fraction of windows with valid JSON
      - y_left_min_values:    list of y_axis_left min per window (plot 0), for drift inspection
      - y_left_max_values:    list of y_axis_left max per window (plot 0)
    """
    valid = [r for r in folder_results if r["output"] and not r["parse_error"]]
    n = len(folder_results)
    parse_success_rate = len(valid) / n if n > 0 else 0.0

    if len(valid) < 2:
        return {
            "parse_success_rate": parse_success_rate,
            "note": "Not enough valid outputs for consistency analysis"
        }

    # --- Structural consistency (plot 0 only, generalise if needed) ---
    def first_plot(r):
        plots = r["output"].get("plots", [])
        return plots[0] if plots else None

    num_plots_list  = [r["output"].get("num_plots") for r in valid]
    num_series_list = [first_plot(r)["num_series"] for r in valid if first_plot(r)]
    dual_y_list     = [first_plot(r)["dual_y_axis"] for r in valid if first_plot(r)]

    # Label and color sets per window (plot 0)
    def label_set(r):
        p = first_plot(r)
        if not p: return frozenset()
        return frozenset(s["label"].lower().strip() for s in p.get("series", []))

    def color_set(r):
        p = first_plot(r)
        if not p: return frozenset()
        return frozenset(s["color"].lower().strip() for s in p.get("series", []))

    label_sets = [label_set(r) for r in valid]
    color_sets = [color_set(r) for r in valid]

    # Mode = most commonly observed set
    mode_labels = Counter(label_sets).most_common(1)[0][0]
    mode_colors = Counter(color_sets).most_common(1)[0][0]

    label_consistency = sum(1 for ls in label_sets if ls == mode_labels) / len(label_sets)
    color_consistency = sum(1 for cs in color_sets if cs == mode_colors) / len(color_sets)

    # Y-axis value sequences (for drift / monotonicity inspection)
    y_left_mins, y_left_maxs = [], []
    for r in valid:
        p = first_plot(r)
        if p and p.get("y_axis_left"):
            y_left_mins.append(p["y_axis_left"].get("min"))
            y_left_maxs.append(p["y_axis_left"].get("max"))

    # Y-axis range drift: std of reported ranges across windows
    # High std = the model is inconsistently reading the axis as the window rolls
    def safe_std(vals):
        vals = [v for v in vals if v is not None]
        if len(vals) < 2: return None
        mean = sum(vals) / len(vals)
        return (sum((v - mean) ** 2 for v in vals) / len(vals)) ** 0.5

    return {
        "parse_success_rate":  round(parse_success_rate, 3),
        "num_plots_stable":    len(set(num_plots_list)) == 1,
        "num_series_stable":   len(set(num_series_list)) == 1,
        "dual_y_stable":       len(set(dual_y_list)) == 1,
        "label_consistency":   round(label_consistency, 3),   # 1.0 = perfect
        "color_consistency":   round(color_consistency, 3),
        "mode_labels":         sorted(mode_labels),
        "mode_colors":         sorted(mode_colors),
        "y_left_min_sequence": y_left_mins,    # inspect for monotonic drift
        "y_left_max_sequence": y_left_maxs,
        "y_left_min_std":      safe_std(y_left_mins),  # None if < 2 values
        "y_left_max_std":      safe_std(y_left_maxs),
        "n_valid":             len(valid),
        "n_total":             n,
    }

# ------------------------------
# PROCESS ALL CHARTS
# ------------------------------
chart_folders = get_chart_folders(ROOT_FOLDER)
print(f"Found {len(chart_folders)} chart groups\n")

results      = {}
consistency  = {}   # rolling window consistency report per chart type

for chart_folder in chart_folders:
    chart_name = os.path.basename(chart_folder)
    print(f"\n📁 Processing: {chart_name}")

    images = get_images(chart_folder)
    print(f"   Found {len(images)} images")

    results[chart_name] = []

    for image_path in images:
        fname = os.path.basename(image_path)
        print(f"   📊 {fname}")

        image_base64 = encode_image(image_path)
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": PROMPT},
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_base64}"}}
                ]
            }]
        )

        raw_output = response.choices[0].message.content
        cleaned = strip_fences(raw_output)
        entry = {"image": fname, "output": None, "parse_error": None, "validation_issues": []}

        try:
            parsed = json.loads(cleaned)
            _, issues = validate_output(parsed)
            entry["output"] = parsed
            entry["validation_issues"] = issues
            if issues:
                print(f"   ⚠️  Schema issues: {issues}")
            dual_count = sum(1 for p in parsed.get("plots", []) if p.get("dual_y_axis"))
            if dual_count:
                print(f"   🔀 {dual_count} subplot(s) with dual Y-axis")
        except json.JSONDecodeError as e:
            print(f"   ❌ JSON parse error: {e}")
            entry["parse_error"] = str(e)
            entry["output"] = raw_output

        results[chart_name].append(entry)

    # --- Compute rolling consistency AFTER all images in this folder ---
    consistency[chart_name] = rolling_consistency(results[chart_name])
    c = consistency[chart_name]
    print(f"\n   📈 Rolling consistency for {chart_name}:")
    print(f"      Parse success:    {c['parse_success_rate']:.0%}")
    if "note" not in c:
        print(f"      Plots stable:     {c['num_plots_stable']}")
        print(f"      Series stable:    {c['num_series_stable']}")
        print(f"      Dual-Y stable:    {c['dual_y_stable']}")
        print(f"      Label consist.:   {c['label_consistency']:.0%}")
        print(f"      Color consist.:   {c['color_consistency']:.0%}")
    print("   ✅ Done")

# ------------------------------
# SAVE RESULTS + CONSISTENCY REPORT
# ------------------------------
with open("llm_outputs_grouped_rolling.json", "w") as f:
    json.dump(results, f, indent=2)

with open("rolling_consistency_report.json", "w") as f:
    json.dump(consistency, f, indent=2)

print("\n✅ Results saved to llm_outputs_grouped_2.json")
print("✅ Consistency report saved to rolling_consistency_report.json")

Found 6 chart groups


📁 Processing: Chart 6
   Found 289 images
   📊 ^GSPC_2025-03-05_chart6.png
   📊 ^GSPC_2025-03-06_chart6.png
   📊 ^GSPC_2025-03-07_chart6.png


KeyboardInterrupt: 

In [ ]:
import base64
import json
from openai import OpenAI

# ------------------------------
# CONFIG
# ------------------------------
API_KEY = ""
BASE_URL = "https://finderllm2-resource.services.ai.azure.com/openai/v1"
MODEL = "gpt-5-mini"

client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

# ------------------------------
# LOAD IMAGE
# ------------------------------
def encode_image(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()

image_path = "GSPC_2025-05-08_chart3.png"
image_base64 = encode_image(image_path)

# ------------------------------
# PROMPT
# ------------------------------
PROMPT = """
You are a vision system analyzing a financial chart.

Describe ONLY what is visually present.

A dual y-axis exists when there are TWO distinct scales: one on the LEFT side and one on the RIGHT side of the same subplot, each with its own label or tick values.

For each subplot:
- number of subplots
- number of time series
- color and label of each series
- which y-axis each series belongs to: "left" or "right"
- x-axis label and visible tick values
- left y-axis: label, approximate min and max
- right y-axis: label, approximate min and max (only if visually present, otherwise null)

Return ONLY valid JSON in this format:

{
  "num_plots": int,
  "plots": [
    {
      "plot_id": int,
      "num_series": int,
      "dual_y_axis": bool,
      "series": [
        {
          "color": "...",
          "label": "...",
          "y_axis_side": "left" or "right"
        }
      ],
      "x_axis": {
        "label": "...",
        "values": []
      },
      "y_axis_left": {
        "label": "...",
        "min": float,
        "max": float
      },
      "y_axis_right": {
        "label": null,
        "min": null,
        "max": null
      }
    }
  ]
}

If a subplot has no right y-axis, set dual_y_axis to false and fill y_axis_right with nulls.
If a subplot has a right y-axis, set dual_y_axis to true and fill both y_axis_left and y_axis_right.
"""

# ------------------------------
# CALL MODEL
# ------------------------------
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": PROMPT},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{image_base64}"
                    }
                }
            ]
        }
    ]
)

# ------------------------------
# PARSE OUTPUT
# Strip markdown fences if the model wraps output in ```json ... ```
# ------------------------------
raw = response.choices[0].message.content.strip()
if raw.startswith("```"):
    raw = raw.split("```")[1]
    if raw.startswith("json"):
        raw = raw[4:]
    raw = raw.strip()

try:
    parsed = json.loads(raw)

    # Quick summary print
    print(f"Plots detected: {parsed['num_plots']}\n")
    for plot in parsed["plots"]:
        dual = plot.get("dual_y_axis", False)
        print(f"  Plot {plot['plot_id']} — {plot['num_series']} series — dual Y-axis: {dual}")
        for s in plot["series"]:
            print(f"    [{s['y_axis_side']}] {s['label']} ({s['color']})")
        ly = plot["y_axis_left"]
        print(f"    Y-left:  {ly['label']}  [{ly['min']} → {ly['max']}]")
        if dual:
            ry = plot["y_axis_right"]
            print(f"    Y-right: {ry['label']}  [{ry['min']} → {ry['max']}]")

    print("\n--- Full JSON ---")
    print(json.dumps(parsed, indent=2))

except json.JSONDecodeError as e:
    print(f"JSON parse error: {e}\n")
    print("Raw output:")
    print(raw)

Plots detected: 3

  Plot 1 — 3 series — dual Y-axis: True
    [left] OBV (blue)
    [right] ADX (red)
    [right] Strong Trend (ADX > 25) (orange)
    Y-left:  OBV  [120000000000.0 → 180000000000.0]
    Y-right: ADX  [10.0 → 60.0]
  Plot 2 — 6 series — dual Y-axis: True
    [left] Skewness (20d) (blue)
    [left] Skewness (60d) (cyan)
    [left] Skewness = 0 (gray)
    [right] Kurtosis (20d) (red)
    [right] Kurtosis (60d) (orange)
    [right] Kurtosis = 0 (gray)
    Y-left:  Skewness  [-2.5 → 2.0]
    Y-right: Kurtosis  [0.0 → 12.0]
  Plot 3 — 4 series — dual Y-axis: False
    [left] 5% VaR (red)
    [left] 1% VaR (brown)
    [left] 5% ES (orange)
    [left] Daily Returns (gray)
    Y-left:  VaR & ES  [-0.06 → 0.08]

--- Full JSON ---
{
  "num_plots": 3,
  "plots": [
    {
      "plot_id": 1,
      "num_series": 3,
      "dual_y_axis": true,
      "series": [
        {
          "color": "blue",
          "label": "OBV",
          "y_axis_side": "left"
        },
        {
         

In [ ]:
import base64
import json
import re
from typing import List, Dict, Any
from openai import OpenAI

# ==============================
# CONFIG
# ==============================
API_KEY = ""
BASE_URL = "https://finderllm2-resource.services.ai.azure.com/openai/v1"
MODEL = "gpt-5-mini"

# IMPORTANT:
# Put the files in the same logical order used below.
# If your actual image order is different, just swap the paths here.
IMAGE_SPECS = [
    {
        "image_id": "A",
        "path": "GSPC_2025-05-08_chart2.png",
        "description": "Price trend / volatility / oscillators / momentum"
    },
    {
        "image_id": "B",
        "path": "GSPC_2025-05-08_chart3.png",
        "description": "OBV / skewness-kurtosis / VaR-ES"
    },
    {
        "image_id": "C",
        "path": "GSPC_2025-05-08_chart1.png",
        "description": "ATR-volume / CMF / stochastic / ulcer-CCI"
    }
]

client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

# ==============================
# IMAGE HELPERS
# ==============================
def encode_image(path: str) -> str:
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()

def guess_mime_type(path: str) -> str:
    p = path.lower()
    if p.endswith(".png"):
        return "image/png"
    if p.endswith(".jpg") or p.endswith(".jpeg"):
        return "image/jpeg"
    return "application/octet-stream"

# ==============================
# BENCHMARK QUESTIONS
# One clear question per panel across 3 images
# ==============================
TEST_ITEMS = [
    {
        "id": 1,
        "image_id": "A",
        "panel": "Price trend",
        "question": "Which moving average is closest to the closing price at the end: SMA20, SMA50, or SMA200?",
        "allowed_answers": ["SMA20", "SMA50", "SMA200"],
        "expected_answer": "SMA50",
        "accepted_answers": ["sma50", "sma 50"]
    },
    {
        "id": 2,
        "image_id": "A",
        "panel": "Volatility",
        "question": "Which volatility measure is highest at the end of the chart?",
        "allowed_answers": ["HV_10", "HV_30", "VOL_PARKINSON", "VOL_GK", "VOL_RS", "COMPOSITE_VOL", "VOLVOFVOL"],
        "expected_answer": "HV_30",
        "accepted_answers": ["hv_30", "hv30", "hv 30"]
    },
    {
        "id": 3,
        "image_id": "A",
        "panel": "Oscillators",
        "question": "At the end of the chart, is RSI above or below the overbought threshold?",
        "allowed_answers": ["Above", "Below"],
        "expected_answer": "Above",
        "accepted_answers": ["above", "above the overbought threshold", "overbought"]
    },
    {
        "id": 4,
        "image_id": "A",
        "panel": "Momentum",
        "question": "Which momentum line is highest at the end: ROC_5, ROC_10, or ROC_20?",
        "allowed_answers": ["ROC_5", "ROC_10", "ROC_20"],
        "expected_answer": "ROC_20",
        "accepted_answers": ["roc_20", "roc20", "roc 20"]
    },
    {
        "id": 5,
        "image_id": "B",
        "panel": "OBV + ADX",
        "question": "At the end of the chart, is ADX above or below the strong-trend threshold?",
        "allowed_answers": ["Above", "Below"],
        "expected_answer": "Below",
        "accepted_answers": ["below", "below the threshold", "below strong trend threshold"]
    },
    {
        "id": 6,
        "image_id": "B",
        "panel": "Skewness",
        "question": "At the end of the chart, is 20-day skewness above or below zero?",
        "allowed_answers": ["Above", "Below"],
        "expected_answer": "Above",
        "accepted_answers": ["above", "above zero", "positive"]
    },
    {
        "id": 7,
        "image_id": "B",
        "panel": "VaR + ES",
        "question": "At the end of the chart, which risk line is lowest: 5% VaR, 1% VaR, or 5% ES?",
        "allowed_answers": ["5% VaR", "1% VaR", "5% ES"],
        "expected_answer": "5% ES",
        "accepted_answers": ["5% es", "es", "5 percent es"]
    },
    {
        "id": 8,
        "image_id": "C",
        "panel": "ATR + Volume",
        "question": "At the end of the chart, is ATR rising or falling?",
        "allowed_answers": ["Rising", "Falling"],
        "expected_answer": "Falling",
        "accepted_answers": ["falling", "declining", "going down"]
    },
    {
        "id": 9,
        "image_id": "C",
        "panel": "CMF",
        "question": "At the end of the chart, is CMF above or below zero?",
        "allowed_answers": ["Above", "Below"],
        "expected_answer": "Above",
        "accepted_answers": ["above", "above zero", "positive"]
    },
    {
        "id": 10,
        "image_id": "C",
        "panel": "Stochastic",
        "question": "At the end of the chart, are Stoch %K and Stoch %D both above 80?",
        "allowed_answers": ["Yes", "No"],
        "expected_answer": "Yes",
        "accepted_answers": ["yes", "both above 80", "above 80"]
    },
    {
        "id": 11,
        "image_id": "C",
        "panel": "CCI",
        "question": "At the end of the chart, is CCI above +100, below -100, or between them?",
        "allowed_answers": ["Above +100", "Below -100", "Between"],
        "expected_answer": "Between",
        "accepted_answers": ["between", "between them", "between the thresholds"]
    }
]

# ==============================
# PROMPT
# ==============================
PROMPT = f"""
You are evaluating THREE financial chart images.

The images are presented in this exact order:
- Image A
- Image B
- Image C

Answer the questions using ONLY the visual information in the images.
Each question explicitly tells you which image/panel it refers to.
Do not use line colors in your reasoning.
Do not infer from finance knowledge beyond what is visible.
If a question asks about "at the end", focus on the last visible plotted point.

Questions:
{json.dumps(TEST_ITEMS, indent=2)}

Return ONLY valid JSON in exactly this format:

{{
  "num_questions": 11,
  "answers": [
    {{
      "id": 1,
      "image_id": "A",
      "panel": "Price trend",
      "question": "Which moving average is closest to the closing price at the end: SMA20, SMA50, or SMA200?",
      "model_answer": "SMA50",
      "justification": "At the right edge, the closing price lies closest to SMA50."
    }}
  ]
}}

Rules:
- Give exactly one answer for each question.
- model_answer must be exactly one of the allowed answers for that question.
- justification must be one short professional sentence.
- Use only visual evidence from the images.
- Do not include markdown fences.
- Do not include any text before or after the JSON.
"""

# ==============================
# OPTIONAL STRUCTURED OUTPUT
# ==============================
RESPONSE_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "multi_image_chart_reasoning_eval",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "num_questions": {"type": "integer"},
                "answers": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "id": {"type": "integer"},
                            "image_id": {"type": "string"},
                            "panel": {"type": "string"},
                            "question": {"type": "string"},
                            "model_answer": {"type": "string"},
                            "justification": {"type": "string"}
                        },
                        "required": ["id", "image_id", "panel", "question", "model_answer", "justification"],
                        "additionalProperties": False
                    }
                }
            },
            "required": ["num_questions", "answers"],
            "additionalProperties": False
        }
    }
}

# ==============================
# HELPERS
# ==============================
def strip_fences(text: str) -> str:
    text = text.strip()
    if text.startswith("```"):
        parts = text.split("```")
        if len(parts) >= 2:
            text = parts[1]
            if text.startswith("json"):
                text = text[4:]
    return text.strip()

def normalize(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\s%+\-_]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def score_answer(model_answer: str, justification: str, expected_answer: str, accepted_answers: List[str]) -> Dict[str, Any]:
    ma = normalize(model_answer)
    ju = normalize(justification)
    combined = f"{ma} {ju}".strip()

    expected = normalize(expected_answer)
    correct = (ma == expected)

    if not correct:
        if expected in combined:
            correct = True

    if not correct:
        for phrase in accepted_answers:
            if normalize(phrase) in combined:
                correct = True
                break

    return {
        "correct": correct,
        "normalized_model_answer": ma,
        "normalized_justification": ju
    }

def build_messages(prompt_text: str, image_specs: List[Dict[str, str]]):
    content = [{"type": "text", "text": prompt_text}]
    for spec in image_specs:
        mime = guess_mime_type(spec["path"])
        img_b64 = encode_image(spec["path"])
        content.append({
            "type": "text",
            "text": f"Image {spec['image_id']} - {spec['description']}"
        })
        content.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:{mime};base64,{img_b64}"
            }
        })
    return [{"role": "user", "content": content}]

# ==============================
# MODEL CALL
# ==============================
messages = build_messages(PROMPT, IMAGE_SPECS)

parsed = None
raw = None
used_structured_output = False
call_mode = None

try:
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        response_format=RESPONSE_SCHEMA
    )
    raw = response.choices[0].message.content
    if isinstance(raw, str):
        parsed = json.loads(raw)
    else:
        parsed = raw
    used_structured_output = True
    call_mode = "json_schema"
except Exception:
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            response_format={"type": "json_object"}
        )
        raw = response.choices[0].message.content
        raw = strip_fences(raw)
        parsed = json.loads(raw)
        used_structured_output = False
        call_mode = "json_object"
    except Exception:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages
        )
        raw = response.choices[0].message.content
        raw = strip_fences(raw)
        parsed = json.loads(raw)
        used_structured_output = False
        call_mode = "plain_prompted_json"

# ==============================
# VALIDATE / SCORE
# ==============================
gold_by_id = {item["id"]: item for item in TEST_ITEMS}

scored_answers = []
num_correct = 0
num_answered = 0

for ans in parsed.get("answers", []):
    qid = ans.get("id")
    if qid not in gold_by_id:
        continue

    gold = gold_by_id[qid]
    model_answer = ans.get("model_answer", "").strip()
    justification = ans.get("justification", "").strip()

    scoring = score_answer(
        model_answer=model_answer,
        justification=justification,
        expected_answer=gold["expected_answer"],
        accepted_answers=gold["accepted_answers"]
    )

    row = {
        "id": qid,
        "image_id": gold["image_id"],
        "panel": gold["panel"],
        "question": gold["question"],
        "allowed_answers": gold["allowed_answers"],
        "expected_answer": gold["expected_answer"],
        "model_answer": model_answer,
        "justification": justification,
        "correct": scoring["correct"]
    }
    scored_answers.append(row)
    num_answered += 1
    num_correct += int(scoring["correct"])

accuracy = num_correct / len(TEST_ITEMS) if TEST_ITEMS else 0.0

report = {
    "model": MODEL,
    "image_paths": [spec["path"] for spec in IMAGE_SPECS],
    "call_mode": call_mode,
    "used_structured_output": used_structured_output,
    "num_questions": len(TEST_ITEMS),
    "num_answered": num_answered,
    "num_correct": num_correct,
    "accuracy": round(accuracy, 4),
    "results": scored_answers
}

# ==============================
# PRINT SUMMARY
# ==============================
print(f"Model: {report['model']}")
print("Images:")
for p in report["image_paths"]:
    print(f"  - {p}")
print(f"Call mode: {report['call_mode']}")
print(f"Structured output used: {report['used_structured_output']}")
print(f"Score: {report['num_correct']}/{report['num_questions']} | Accuracy: {report['accuracy']:.2%}")
print()

for row in report["results"]:
    mark = "OK" if row["correct"] else "FAIL"
    print(f"[{mark}] Q{row['id']} | Image {row['image_id']} | {row['panel']}")
    print(f"  Question: {row['question']}")
    print(f"  Allowed:  {row['allowed_answers']}")
    print(f"  Expected: {row['expected_answer']}")
    print(f"  Model:    {row['model_answer']}")
    print(f"  Why:      {row['justification']}")
    print()

print("--- Full JSON report ---")
print(json.dumps(report, indent=2))

Model: gpt-5-mini
Images:
  - GSPC_2025-05-08_chart2.png
  - GSPC_2025-05-08_chart3.png
  - GSPC_2025-05-08_chart1.png
Call mode: json_schema
Structured output used: True
Score: 11/11 | Accuracy: 100.00%

[OK] Q1 | Image A | Price trend
  Question: Which moving average is closest to the closing price at the end: SMA20, SMA50, or SMA200?
  Allowed:  ['SMA20', 'SMA50', 'SMA200']
  Expected: SMA50
  Model:    SMA50
  Why:      At the right edge the closing price is visually nearest the SMA50 line compared with the SMA20 and SMA200.

[OK] Q2 | Image A | Volatility
  Question: Which volatility measure is highest at the end of the chart?
  Allowed:  ['HV_10', 'HV_30', 'VOL_PARKINSON', 'VOL_GK', 'VOL_RS', 'COMPOSITE_VOL', 'VOLVOFVOL']
  Expected: HV_30
  Model:    HV_30
  Why:      At the final date the HV_30 trace is above the other volatility measures on the volatility panel.

[OK] Q3 | Image A | Oscillators
  Question: At the end of the chart, is RSI above or below the overbought threshold